In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset=CIFAR10(root='./data',train=True,download=True,transform=transform)
testset=CIFAR10(root='./data',train=False,download=True,transform=transform)

trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

100%|██████████| 170M/170M [23:16<00:00, 122kB/s]  


In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        # Convolutional layer
        self.conv_layers=nn.Sequential(
            # 1sst layer
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            # 2nd layer
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            # 3rd layer
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), 
        )
        # Fully Connected layer
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x     

In [19]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [21]:
# Training the Model
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for images,labels in trainloader:
        optimizer.zero_grad()
        output=model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch+1}/epochs & loss={epoch_training_loss/len(trainloader)}")

epoch=1/epochs & loss=0.9567942509565817
epoch=2/epochs & loss=0.763059318332416
epoch=3/epochs & loss=0.6324495278546572
epoch=4/epochs & loss=0.5286447587030013
epoch=5/epochs & loss=0.43139548740728434
epoch=6/epochs & loss=0.3521733105830524
epoch=7/epochs & loss=0.2764329626641767
epoch=8/epochs & loss=0.21816582676699703
epoch=9/epochs & loss=0.16934894141086074
epoch=10/epochs & loss=0.1431731119218385


In [26]:
# Evaluating the CNN Model
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)
        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f"Accuracy={correct_labels/total_labels *100}")

Accuracy=74.85000000000001


# 